# Transcricao PT-BR -> legenda .srt (faster-whisper large-v3 + revisao com Qwen3)

**Fluxo:** o video `.mp4` vira `.mp3` **no seu PC** (`mp4_para_mp3.py`).
Aqui no Colab entra **so o .mp3** - nenhum frame de video sobe para a nuvem.

**Para rodar:** menu `Ambiente de execucao` > `Executar tudo`.
Este notebook ja pede a **GPU T4** sozinho. Se mesmo assim a celula 3 avisar que falta GPU:
`Ambiente de execucao` > `Alterar o tipo de ambiente de execucao` > **T4 GPU** > `Salvar`.

**Saem dois arquivos, nessa ordem:**
1. `seu_audio_bruto.srt` - a transcricao crua, salva assim que a celula 4 termina;
2. `seu_audio.srt` - o mesmo texto depois da revisao de portugues da celula 5. **E este que voce usa.**

A revisao roda sozinha logo depois, na mesma GPU: um modelo de linguagem (Qwen3-8B) le so o
*texto* das legendas - nunca os tempos - e devolve pontuacao, acentos, crase e nomes proprios
arrumados. Ele nao pode reescrever, juntar, dividir nem inventar frase. Para desligar:
`POLIR = False` no topo da celula 5.

Configuracao fixa deste notebook:
- modelo `large-v3`, idioma travado em portugues (`language="pt"`);
- VAD ligado (corta silencio, reduz alucinacao do modelo);
- timestamps por palavra -> legendas curtas (max. 2 linhas de 40 caracteres);
- **a legenda 1 comeca em `00:00:00,000`** - a nao ser que o video abra com vinheta ou musica
  longa, e ai a legenda 1 espera a fala comecar de verdade;
- os `.srt` baixam sozinhos no fim.

> Arquivo grande (mais de ~100 MB)? Arraste o `.mp3` para o painel **Arquivos** (icone de pasta
> na barra esquerda) antes de rodar a celula 2 - ela usa o mp3 que ja estiver em `/content`.

> Se aparecer o aviso **"voce esta conectado a uma GPU mas nao esta usando"**, feche e ignore.
> Aceitar a troca reinicia o ambiente e apaga o mp3 que voce subiu.


In [ ]:
#@title 1. Ambiente: GPU + dependencias { display-mode: "form" }
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "SEM GPU -> Ambiente de execucao > Alterar o tipo > T4 GPU"

# ctranslate2 >= 4.6.3 e o que evita o crash de cuDNN no Colab (esses wheels sao compilados
# com WITH_CUDNN=OFF). O ffmpeg nao entra aqui: o faster-whisper decodifica com PyAV.
!pip -q install "faster-whisper==1.2.1" "ctranslate2>=4.6.3"

import ctranslate2, faster_whisper, torch
print("faster-whisper:", faster_whisper.__version__,
      "| ctranslate2:", ctranslate2.__version__,
      "| CUDA:", torch.cuda.is_available())


In [ ]:
#@title 2. Enviar o .mp3 gerado no seu PC { display-mode: "form" }
from pathlib import Path
from google.colab import files

# Ja tem um .mp3 em /content (painel Arquivos ou Drive)? usa ele. Senao, abre o seletor.
existentes = sorted(Path("/content").glob("*.mp3"))
if len(existentes) > 1:
    print("varios mp3 em /content:", ", ".join(p.name for p in existentes))
    print("-> vou usar o primeiro. Apague os outros no painel Arquivos se nao for esse.\n")
if existentes:
    ENTRADA = existentes[0]
    print("usando o mp3 que ja esta no Colab:", ENTRADA.name)
else:
    ENTRADA = Path(next(iter(files.upload())))

print(f"arquivo: {ENTRADA.name}  ({ENTRADA.stat().st_size/1e6:.1f} MB)")

In [ ]:
#@title 3. Transcrever com large-v3 travado em portugues { display-mode: "form" }
import time

import torch
from faster_whisper import WhisperModel

if not torch.cuda.is_available():
    raise SystemExit(
        "SEM GPU. Menu Ambiente de execucao > Alterar o tipo de ambiente de execucao > T4 GPU,"
        " depois Ambiente de execucao > Executar tudo."
    )

# O .mp3 vai direto para o modelo: o faster-whisper decodifica com PyAV e ja reamostra para
# 16 kHz mono por dentro - converter para WAV antes seria so uma etapa a mais para dar errado.
#
# min_silence_duration_ms=2000 e o padrao da lib: com os 500 antigos, o Silero fundia dois
# trechos sempre que o intervalo entre eles era menor que 2*speech_pad_ms (800 ms), e a pausa
# sumia - justamente as pausas de 0,5 a 1,5 s que o GAP_QUEBRA da celula 4 usa para quebrar.
# float16 e o formato nativo da T4: melhor qualidade, e os 3 GB do modelo cabem folgado
# nos 16 GB da placa - quantizar para int8 aqui nao compra nada.
modelo = WhisperModel("large-v3", device="cuda", compute_type="float16")

gerador, info = modelo.transcribe(
    str(ENTRADA),
    language="pt",                 # travado: nao tenta detectar outro idioma
    task="transcribe",
    beam_size=5,
    word_timestamps=True,          # necessario para as legendas curtas
    vad_filter=True,               # corta silencio -> menos alucinacao
    vad_parameters=dict(
        threshold=0.5,                  # o quanto o VAD precisa ter certeza de que aquilo e fala
        min_silence_duration_ms=2000,   # so corta o audio em silencios de 2 s+ (ver comentario acima)
        speech_pad_ms=400,              # nao reduza: com 200 ms o modelo cortava a primeira palavra
    ),
    condition_on_previous_text=False,   # evita o modelo repetir frases em loop
    # a escada precisa terminar em 1.0: se a janela ainda falha no ultimo degrau, o
    # faster-whisper ACEITA o melhor dos fracassos e o loop de repeticao entra no .srt.
    temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
)
print(f"duracao do audio: {info.duration/60:.1f} min - transcrevendo...\n")

t0 = time.time()
segmentos = []
for seg in gerador:                # o gerador so processa quando iterado
    segmentos.append(seg)
    pct = 100 * seg.end / info.duration
    print(f"[{pct:5.1f}%] [{seg.start:7.1f}s] {seg.text.strip()[:80]}")
print(f"\n{len(segmentos)} segmentos em {time.time()-t0:.0f}s")


In [ ]:
#@title 4. Montar as legendas e ja salvar o .srt bruto { display-mode: "form" }
# Regras de exibicao da legenda (padrao proximo ao de streaming)
MAX_CHARS = 40          # caracteres por linha (max_line_width)
MAX_LINHAS = 2          # linhas por legenda (max_line_count)
MAX_DUR = 6.0           # segundos por legenda
MIN_DUR = 0.7           # nenhuma legenda pisca mais rapido que isso
GAP_QUEBRA = 0.7        # silencio entre palavras que forca nova legenda
ESPERA_ANCORA = 6.0     # so puxa a legenda 1 para o zero se a fala comecar ate aqui
FIM_FRASE = ".?!:;"     # pontuacao que fecha a legenda


def coletar_palavras(segmentos):
    """Achata os segmentos em uma lista unica de palavras com tempo."""
    palavras = []
    for seg in segmentos:
        if seg.words:
            for w in seg.words:
                texto = w.word.strip()
                if texto:
                    palavras.append({"t": texto, "ini": w.start, "fim": w.end})
        else:  # segmento sem timestamp por palavra: entra inteiro
            texto = seg.text.strip()
            if texto:
                palavras.append({"t": texto, "ini": seg.start, "fim": seg.end})
    return palavras


def _fatiar(palavra):
    """Palavra maior que a linha inteira (URL, nome tecnico): corta em pedacos."""
    if len(palavra) <= MAX_CHARS:
        return [palavra]
    return [palavra[i:i + MAX_CHARS] for i in range(0, len(palavra), MAX_CHARS)]


def _envolver(texto):
    """Quebra gulosa: linhas de no maximo MAX_CHARS caracteres."""
    linhas, atual = [], ""
    for bruta in texto.split():
        for palavra in _fatiar(bruta):
            if not atual:
                atual = palavra
            elif len(atual) + 1 + len(palavra) <= MAX_CHARS:
                atual += " " + palavra
            else:
                linhas.append(atual)
                atual = palavra
    if atual:
        linhas.append(atual)
    return linhas


def cabe(texto):
    """O texto cabe na tela dentro do limite de linhas?"""
    return len(_envolver(texto)) <= MAX_LINHAS


def quebrar_linhas(texto):
    """Divide em linhas equilibradas (visual melhor que a quebra gulosa)."""
    if len(texto) <= MAX_CHARS:
        return texto
    palavras = texto.split()
    melhor, dif_melhor = None, None
    for corte in range(1, len(palavras)):
        a, b = " ".join(palavras[:corte]), " ".join(palavras[corte:])
        if len(a) > MAX_CHARS or len(b) > MAX_CHARS:
            continue
        dif = abs(len(a) - len(b))
        if dif_melhor is None or dif < dif_melhor:
            melhor, dif_melhor = (a, b), dif
    if melhor:
        return melhor[0] + "\n" + melhor[1]
    return "\n".join(_envolver(texto))


def agrupar(palavras):
    """Junta palavras em blocos de legenda respeitando tela, duracao e pausas."""
    blocos, atual = [], []

    for p in palavras:
        if atual:
            candidato = " ".join(x["t"] for x in atual) + " " + p["t"]
            estourou = (
                not cabe(candidato)
                or p["fim"] - atual[0]["ini"] > MAX_DUR
                or p["ini"] - atual[-1]["fim"] > GAP_QUEBRA
            )
            if estourou:
                blocos.append(atual)
                atual = []
        atual.append(p)
        texto_atual = " ".join(x["t"] for x in atual)
        if p["t"][-1] in FIM_FRASE and len(texto_atual) > MAX_CHARS:
            blocos.append(atual)
            atual = []

    if atual:
        blocos.append(atual)
    return blocos


def montar_legendas(blocos, ancorar_no_zero=True):
    """Blocos -> lista de legendas com tempos limpos, sem sobreposicao."""
    legendas = []
    for bloco in blocos:
        texto = " ".join(p["t"] for p in bloco)
        legendas.append({"ini": bloco[0]["ini"], "fim": bloco[-1]["fim"], "txt": texto})

    if not legendas:
        return legendas

    # A legenda 1 comeca em 00:00:00,000 - a menos que o video abra com vinheta ou
    # musica longa: esticar a legenda por 20 s dava spoiler e ficava fora de sincronia.
    if ancorar_no_zero and legendas[0]["ini"] <= ESPERA_ANCORA:
        legendas[0]["ini"] = 0.0

    anterior_fim = 0.0
    for leg in legendas:
        leg["ini"] = round(max(leg["ini"], anterior_fim), 3)
        fim = max(leg["fim"], leg["ini"] + MIN_DUR)
        leg["fim"] = round(min(fim, leg["ini"] + MAX_DUR), 3)   # nada fica na tela alem do limite
        anterior_fim = leg["fim"]
    return legendas


def tempo_srt(t):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def escrever_srt(legendas, caminho):
    linhas = []
    for i, leg in enumerate(legendas, 1):
        linhas.append(str(i))
        linhas.append(f"{tempo_srt(leg['ini'])} --> {tempo_srt(leg['fim'])}")
        linhas.append(quebrar_linhas(leg["txt"]))
        linhas.append("")
    with open(caminho, "w", encoding="utf-8") as f:
        f.write("\n".join(linhas))
    return caminho


def conferir_invariantes(legendas, inicio_da_fala):
    """Os quatro invariantes do .srt. Vale para o arquivo bruto e para o revisado."""
    # 1. legenda 1 no zero, salvo vinheta longa no comeco do video
    if inicio_da_fala <= ESPERA_ANCORA:
        assert tempo_srt(legendas[0]["ini"]) == "00:00:00,000", "ancoragem no zero falhou"
    # 2. nenhuma legenda comeca antes do fim da anterior
    assert all(b["ini"] >= a["fim"] - 1e-3 for a, b in zip(legendas, legendas[1:])), \
        "legendas sobrepostas"
    # 3. duracao dentro da faixa
    assert all(MIN_DUR - 1e-3 <= leg["fim"] - leg["ini"] <= MAX_DUR + 1e-3 for leg in legendas), \
        "duracao fora de MIN_DUR/MAX_DUR"
    # 4. no maximo MAX_LINHAS linhas de MAX_CHARS caracteres (vale depois da revisao tambem)
    for leg in legendas:
        linhas = quebrar_linhas(leg["txt"]).split("\n")
        assert len(linhas) <= MAX_LINHAS, f"mais de {MAX_LINHAS} linhas: {leg['txt']!r}"
        assert max(len(x) for x in linhas) <= MAX_CHARS, f"linha > {MAX_CHARS}: {leg['txt']!r}"


palavras = coletar_palavras(segmentos)
legendas = montar_legendas(agrupar(palavras), ancorar_no_zero=True)
# Copia crua do texto: e o que deixa a celula 5 ser re-executavel sem revisar o ja revisado.
TEXTO_BRUTO = [leg["txt"] for leg in legendas]
inicio_da_fala = min(p["ini"] for p in palavras)

conferir_invariantes(legendas, inicio_da_fala)
# Resultado 1 de 2: sai agora, sem esperar o LLM. Se a revisao falhar ou o Colab cair,
# este arquivo ja esta pronto no painel Arquivos.
SRT_BRUTO = escrever_srt(legendas, ENTRADA.with_name(ENTRADA.stem + "_bruto.srt"))

print(f"{len(palavras)} palavras -> {len(legendas)} legendas")
print("primeira legenda comeca em:", tempo_srt(legendas[0]["ini"]))
if inicio_da_fala > ESPERA_ANCORA:
    print(f"(a fala so comeca em {tempo_srt(inicio_da_fala)} - vinheta/musica no inicio,"
          " entao a legenda 1 nao foi puxada para o zero)")
print("legenda crua salva em:", SRT_BRUTO.name)


In [ ]:
#@title 5. Revisar a pontuacao com um LLM na mesma GPU { display-mode: "form" }
POLIR = True             # False = pula a revisao; o .srt final sai igual ao bruto

LOTE = 25                # legendas enviadas de uma vez ao modelo
CONTEXTO = 3             # legendas anteriores mandadas so como leitura (nao voltam)
CRESCIMENTO_MAX = 1.3    # o texto revisado nao pode passar disso do tamanho original
REPO_GGUF = "Qwen/Qwen3-8B-GGUF"
ARQUIVO_GGUF = "Qwen3-8B-Q4_K_M.gguf"   # ~5 GB; sobra VRAM na T4 depois de soltar o whisper

if POLIR:
    # Wheel ja compilado com CUDA (cu124 e o indice mais novo publicado pelo projeto):
    # compilar do fonte dentro do Colab passa de 20 min.
    !pip -q install "llama-cpp-python==0.3.19" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import json

# O modelo nunca ve tempo: recebe indice + texto e devolve indice + texto. Os tempos sao
# remontados aqui, a partir da lista original - por isso os invariantes 1 a 3 nao correm risco.
# Esquema chapado de proposito (so integer e string): a conversao para GBNF do llama.cpp
# quebra com atalhos de regex tipo \d, \w, \s.
ESQUEMA = {
    "type": "object",
    "properties": {
        "legendas": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"i": {"type": "integer"}, "t": {"type": "string"}},
                "required": ["i", "t"],
            },
        }
    },
    "required": ["legendas"],
}

# Unico texto acentuado do projeto, de proposito: isto e material linguistico para o modelo,
# nao saida de console. Escrever "pontuacao" aqui seria ensinar ao revisor o erro que ele tem
# que corrigir. O "/no_think" desliga o modo de raciocinio do Qwen3.
REGRAS = """/no_think
Você revisa legendas de vídeo em português do Brasil. O texto veio de um reconhecimento
automático de fala e pode ter erros.

CORRIJA apenas:
- ponto final que falta no fim da frase; vírgula faltando ou sobrando; dois-pontos faltando
- trecho longo sem nenhuma pontuação: separe com pontuação, sem reescrever as palavras
- letra minúscula depois de ponto final; maiúscula indevida depois de vírgula
- concordância verbal e nominal
- regência e crase (a / à / há)
- palavra trocada ou inexistente, típica de erro de transcrição
- homófonos confundidos: sessão / seção / cessão, mas / mais, mau / mal, porque / por que
- títulos, siglas e nomes próprios: capitalização correta
- espaço sobrando antes de hífen; travessão de diálogo no início da linha

NUNCA:
- reescrever o estilo, trocar palavra por sinônimo ou "melhorar" a frase
- juntar duas legendas, dividir uma legenda, mudar a ordem ou a quantidade delas
- traduzir, resumir, comentar ou explicar
- inventar conteúdo: você não tem o áudio. Legenda que termina no meio da frase continua
  terminando no meio da frase - ajuste só a pontuação
- devolver um texto bem maior que o original

FORMATO: responda só com o JSON pedido. Em "legendas", devolva exatamente as mesmas legendas
do campo "revisar", na mesma ordem e com os mesmos índices "i". Legenda que já está certa volta
com o texto idêntico. O campo "contexto" é só leitura: nunca devolva essas legendas."""


def _pedido(lote, contexto):
    """Mensagem do usuario: contexto so para leitura + as legendas a revisar."""
    return json.dumps(
        {
            "contexto": [{"i": i, "t": t} for i, t in contexto],
            "revisar": [{"i": i, "t": t} for i, t in lote],
        },
        ensure_ascii=False,
    )


def _extrair(resposta, lote):
    """Valida o lote inteiro. Devolve {indice: texto} ou None se qualquer coisa nao bater."""
    try:
        dados = json.loads(resposta["choices"][0]["message"]["content"])
    except (KeyError, IndexError, TypeError, ValueError):
        return None
    itens = dados.get("legendas")
    if not isinstance(itens, list) or len(itens) != len(lote):   # contagem tem que bater
        return None
    novos = {}
    for item in itens:
        if not isinstance(item, dict):
            return None
        i, t = item.get("i"), item.get("t")
        if isinstance(i, bool) or not isinstance(i, int) or not isinstance(t, str):
            return None
        novos[i] = " ".join(t.split())     # sem quebra de linha: o .srt requebra sozinho
    if set(novos) != {i for i, _ in lote}:  # indice trocado, faltando ou repetido
        return None
    return novos


def _aceitar(velho, novo):
    """Guarda por legenda: nada de texto inchado nem de legenda que estoura a tela."""
    return bool(novo) and len(novo) <= CRESCIMENTO_MAX * len(velho) and cabe(novo)


if not POLIR:
    print("POLIR = False -> o .srt final vai sair igual ao bruto.")
else:
    import gc
    import time

    import torch
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    # A T4 tem 16 GB, mas o large-v3 ainda ocupa ~3 GB: solta antes de carregar o Qwen3.
    for _nome in ("modelo", "gerador"):
        if _nome in globals():
            del globals()[_nome]
    gc.collect()
    torch.cuda.empty_cache()

    caminho_gguf = hf_hub_download(repo_id=REPO_GGUF, filename=ARQUIVO_GGUF)
    llm = Llama(
        model_path=caminho_gguf,
        n_ctx=4096,          # lote de 25 legendas + contexto + resposta cabem folgado
        n_gpu_layers=-1,     # todas as camadas na GPU
        n_batch=512,
        verbose=False,
    )
    print("modelo carregado | chat_format:", llm.chat_format)

    # Re-rodar esta celula nao empilha revisao em cima de revisao.
    for leg, bruto in zip(legendas, TEXTO_BRUTO):
        leg["txt"] = bruto

    indices = list(range(len(legendas)))
    lotes = [indices[k:k + LOTE] for k in range(0, len(indices), LOTE)]
    t0, ajustadas, descartadas = time.time(), 0, 0

    for n, bloco in enumerate(lotes, 1):
        lote = [(i, legendas[i]["txt"]) for i in bloco]
        contexto = [(i, legendas[i]["txt"]) for i in range(max(0, bloco[0] - CONTEXTO), bloco[0])]
        pedido = _pedido(lote, contexto)

        novos = None
        for temperatura in (0.0, 0.3):   # uma repeticao so, com a temperatura mexida
            resposta = llm.create_chat_completion(
                messages=[
                    {"role": "system", "content": REGRAS},
                    {"role": "user", "content": pedido},
                ],
                response_format={"type": "json_object", "schema": ESQUEMA},  # GBNF nativo
                temperature=temperatura,
                top_p=0.8,
                repeat_penalty=1.0,   # 1.1 (padrao) penaliza as chaves "i"/"t" repetidas do JSON
                max_tokens=2048,
            )
            novos = _extrair(resposta, lote)
            if novos is not None:
                break

        pct = 100 * n // len(lotes)
        if novos is None:            # lote inteiro descartado: fica o texto cru
            descartadas += len(bloco)
            print(f"[{pct:3d}%] lote {n}/{len(lotes)}: resposta invalida duas vezes, mantido o original")
            continue

        mudou = 0
        for i in bloco:
            novo = novos[i]
            if novo != legendas[i]["txt"] and _aceitar(legendas[i]["txt"], novo):
                legendas[i]["txt"] = novo
                mudou += 1
        ajustadas += mudou
        print(f"[{pct:3d}%] lote {n}/{len(lotes)}: {mudou} de {len(bloco)} legendas ajustadas")

    print(f"\nrevisao em {time.time()-t0:.0f}s - {ajustadas} legendas ajustadas,"
          f" {descartadas} mantidas como estavam")
    del llm
    gc.collect()


In [ ]:
#@title 6. Escrever o .srt final (revisado) { display-mode: "form" }
# quebrar_linhas() roda de novo aqui dentro do escrever_srt: o texto revisado e requebrado
# do zero, entao o limite de 2 linhas x 40 caracteres continua valendo.
conferir_invariantes(legendas, inicio_da_fala)
SRT = escrever_srt(legendas, ENTRADA.with_suffix(".srt"))

print(f"{len(legendas)} legendas conferidas (ancora, sobreposicao, duracao, largura de tela)")
print("arquivo bruto :", SRT_BRUTO.name)
print("arquivo final :", SRT.name, "(revisado)" if POLIR else "(igual ao bruto - POLIR = False)")


In [ ]:
#@title 7. Conferir e baixar os dois arquivos .srt { display-mode: "form" }
import time

from google.colab import files

print("=== FINAL (revisado) ===")
print(SRT.read_text(encoding="utf-8")[:600], "...\n")

print("Dois arquivos ficaram no painel Arquivos:")
print(f"  {SRT.name}        <- use este")
print(f"  {SRT_BRUTO.name}  <- a transcricao crua, para comparar")
print("Se algum download nao comecar sozinho: painel Arquivos (icone de pasta, a esquerda),"
      " botao direito no arquivo, Fazer download.")

files.download(str(SRT))
time.sleep(2)          # o navegador ignora o segundo download se os dois saem juntos
files.download(str(SRT_BRUTO))
